# 6. Figuras del trabajo de grado

Este notebook produce las figuras del documento que no son mapas. Los mapas (Figuras 1 y 2) se hacen en ArcGIS Pro.

Cada figura tiene su propia sección, en el orden en que aparece en el documento. Todas comparten el mismo estilo: 16,5 cm de ancho, que es el ancho de texto del documento; letra sin serifa de 8 a 9 pt, como recomienda APA 7, y exportación en PNG a 300 ppp y en PDF.

**Cómo usarlo.** Se trabaja sobre la misma carpeta de Google Drive de los notebooks 1 a 5. Ejecuta primero la sección 6.1 y luego solo la sección de la figura que necesites. Cada figura se guarda en `salidas/figuras_tdg/`.

No entrena ni modifica nada: solo lee `dataset/` y los archivos de `salidas/`.

## 6.1 Configuración

In [ ]:
from pathlib import Path
import os, sys, hashlib, json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

CARPETA_DRIVE = "MyDrive/Geologia_SCD"   # la misma carpeta de los notebooks 1 a 5
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir(f"/content/drive/{CARPETA_DRIVE}")

DATASET_DIR = Path("dataset")
OUT_DIR = Path("salidas")
FIG_DIR = OUT_DIR / "figuras_tdg"
FIG_DIR.mkdir(parents=True, exist_ok=True)
SEMILLA = 42

# ---------- Estilo común a todas las figuras ----------
CM = 1 / 2.54                      # matplotlib mide en pulgadas
ANCHO = 16.5 * CM                  # ancho de texto del documento
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Liberation Sans", "DejaVu Sans"],
    "font.size": 8, "axes.titlesize": 9, "axes.labelsize": 8,
    "xtick.labelsize": 7.5, "ytick.labelsize": 7.5, "legend.fontsize": 7.5,
    "axes.spines.top": False, "axes.spines.right": False,
    "savefig.dpi": 300, "savefig.bbox": "tight", "savefig.pad_inches": 0.02,
    "figure.dpi": 110,
})
# Paleta Okabe-Ito (legible con daltonismo y en gris), la misma de los mapas de ArcGIS
OKABE = {"azul": "#0072B2", "naranja": "#E69F00", "bermellon": "#D55E00", "verde": "#009E73",
         "celeste": "#56B4E9", "amarillo": "#F0E442", "purpura": "#CC79A7", "gris": "#7F7F7F"}
# Nombres de las clases tal como van en el documento
NOMBRE = {"H2": "SCD", "Jaguey": "Jagüey"}
def nombre(clase):
    return NOMBRE.get(clase, clase)

from matplotlib.ticker import FuncFormatter
# Coma decimal en los ejes, como en el texto del documento
coma_decimal = FuncFormatter(lambda v, _: f"{v:g}".replace(".", ","))

def guardar(fig, nombre_archivo):
    """Guarda la figura en PNG (300 ppp) y en PDF dentro de salidas/figuras_tdg/."""
    for ext in ("png", "pdf"):
        fig.savefig(FIG_DIR / f"{nombre_archivo}.{ext}")
    print(f"Guardada: {FIG_DIR / nombre_archivo}.png y .pdf")

# ---------- Partición del notebook 1 (con la misma comprobación) ----------
def hash_particion(d):
    cols = ["id", "clase", "grupo_particion", "split", "fold_val"]
    texto = d[cols].sort_values("id").to_csv(sep=";", index=False, lineterminator="\n", na_rep="")
    return hashlib.sha256(texto.encode("utf-8")).hexdigest()

part = pd.read_csv(OUT_DIR / "particion.csv", sep=";", encoding="utf-8-sig", dtype={"fold_val": "Int64"})
assert hash_particion(part) == (OUT_DIR / "particion.hash").read_text().strip(), "particion.csv fue modificado."
part["ruta"] = [DATASET_DIR / c / f"{i}.png" for c, i in zip(part["carpeta"], part["id"])]
print(f"Partición verificada: {len(part)} imágenes.")

## 6.2 Figura 3: una imagen de cada clase del conjunto de entrenamiento

Muestra cómo se ven las diez clases en las imágenes que produjo la rutina de adquisición (sección 3.1.3). Las clases SCD y Jagüey van primero, porque son las que el modelo debe separar en la Alta Guajira. Debajo de cada imagen van su nivel de *zoom* y el lado del terreno que cubre, porque la misma geoforma se ve muy distinta con un *zoom* u otro.

Las imágenes se eligen al azar entre las de validación cruzada, con semilla fija. Si alguna no representa bien su clase, escribe en `ELEGIDAS` el `id` que prefieras para esa clase (por ejemplo, `"H2": 263`) y vuelve a ejecutar la celda. La celda imprime los `id` usados, para que puedas citarlos en la nota de la figura.

In [ ]:
ORDEN = ["H2", "Jaguey", "Fairy circle", "Fairy fort", "Farm circle",
         "Flooded dune", "Impact crater", "Karst", "Salt lake", "Thermokarst"]
ELEGIDAS = {}          # por ejemplo: {"H2": 263, "Jaguey": 571}

rng = np.random.default_rng(SEMILLA)
cv = part[part["split"] == "cv"]
filas = []
for clase in ORDEN:
    if clase in ELEGIDAS:
        filas.append(part[part["id"] == ELEGIDAS[clase]].iloc[0])
    else:
        sub = cv[cv["clase"] == clase]
        filas.append(sub.iloc[int(rng.integers(len(sub)))])

fig, ejes = plt.subplots(2, 5, figsize=(ANCHO, ANCHO * 0.50))
for ax, f in zip(ejes.ravel(), filas):
    with Image.open(f["ruta"]) as im:
        im = im.convert("RGB"); im.thumbnail((600, 600))
        ax.imshow(im)
    ax.set_xticks([]); ax.set_yticks([])
    for lado in ax.spines.values():
        lado.set_visible(True); lado.set_linewidth(0.4)
    ax.set_title(nombre(f["clase"]), fontsize=8.5, fontweight="bold", pad=3)
    ax.set_xlabel(f"zoom {int(f['zoom'])} · {f['huella_lado_m']:,.0f} m".replace(",", "."), fontsize=7, labelpad=2)
fig.tight_layout(w_pad=0.4, h_pad=0.6)
guardar(fig, "figura_03_clases")
plt.show()
print("id usados:", {nombre(f['clase']): int(f['id']) for f in filas})

**En Word.** Inserta `figura_03_clases.png` en línea con el texto, centrada y a 16,5 cm de ancho. El rótulo va encima (**Figura 3**, Mayús + Enter, título en cursiva). Una propuesta de título y de nota:

- *Ejemplos de las diez clases del conjunto de entrenamiento*
- *Nota.* Una imagen por clase, tomada de la validación cruzada. Debajo de cada imagen se indican su nivel de *zoom* y el lado del terreno que cubre. Imágenes de Google Maps Static API (Google, 2026); coordenadas de las nueve clases de Ginzburg et al. (2025) según Geymond (2025) y de la clase Jagüey según Lerma et al. (2025).

## 6.3 Figura 4: cómo se formaron los sitios y a dónde fue cada imagen

Esquema de la partición de la sección 3.2.1, en tres paneles:

- **A1.** Primer paso: las imágenes de una clase cuyos centros están a menos de 1 km se unen en cadena en un mismo sitio.
- **A2.** Segundo paso: dos sitios cuyas huellas se superponen se fusionan, aunque sus centros estén a más de 1 km.
- **B.** Destino de las imágenes: el *holdout* queda apartado y la validación cruzada rota cinco veces el pliegue de validación; al final, el modelo final se entrena con los cinco pliegues y se evalúa una vez con el *holdout*.

Los paneles A1 y A2 son esquemas, no datos. El número de imágenes de cada pliegue y del *holdout* se lee de `particion.csv`.

In [ ]:
from matplotlib.patches import Rectangle, Circle, Patch

C_ENT, C_VAL, C_HO = OKABE["azul"], OKABE["verde"], OKABE["naranja"]
tam = part[part["split"] == "cv"]["fold_val"].value_counts().sort_index()
n_ho = int((part["split"] == "holdout").sum())
k_pl = len(tam)

fig = plt.figure(figsize=(ANCHO, ANCHO * 0.46))
gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 1.45], wspace=0.15)

# ---- A1: cadenas a menos de 1 km ----
ax = fig.add_subplot(gs[0]); ax.set_aspect("equal"); ax.axis("off")
grupos = {"#0072B2": [(0.6, 1.1), (1.3, 1.5), (2.0, 1.2)], "#D55E00": [(3.6, 2.9)], "#CC79A7": [(1.0, 3.3), (1.6, 3.7)]}
for c, P in grupos.items():
    for x, y in P:
        ax.add_patch(Circle((x, y), 0.5, fc=c, alpha=0.12, ec=c, lw=0.6, ls="--"))
        ax.plot(x, y, "o", color=c, ms=4)
    for (x1, y1), (x2, y2) in zip(P, P[1:]):
        ax.plot([x1, x2], [y1, y2], "-", color=c, lw=1.1)
ax.annotate("", xy=(2.5, 0.4), xytext=(1.5, 0.4), arrowprops=dict(arrowstyle="<->", lw=0.7))
ax.text(2.0, 0.2, "1 km", ha="center", va="top", fontsize=7)
ax.set_xlim(0, 4.4); ax.set_ylim(-0.3, 5.0)
titulos = ["A1. Cercanía: cadenas\na menos de 1 km"]
ejes_fig = [ax]

# ---- A2: superposición de huellas ----
ax = fig.add_subplot(gs[1]); ax.set_aspect("equal"); ax.axis("off")
cuadros = [(1.3, 2.0, 2.2, "#0072B2"), (3.1, 2.5, 2.2, "#D55E00")]
for cx, cy, L, c in cuadros:
    ax.add_patch(Rectangle((cx - L / 2, cy - L / 2), L, L, fc=c, alpha=0.15, ec=c, lw=0.9))
    ax.plot(cx, cy, "o", color=c, ms=4)
(x1, y1, L1, _), (x2, y2, L2, _) = cuadros
izq, der = max(x1 - L1 / 2, x2 - L2 / 2), min(x1 + L1 / 2, x2 + L2 / 2)
aba, arr = max(y1 - L1 / 2, y2 - L2 / 2), min(y1 + L1 / 2, y2 + L2 / 2)
ax.add_patch(Rectangle((izq, aba), der - izq, arr - aba, fc="#444444", alpha=0.30, ec="none"))
ax.annotate("", xy=(3.1, 0.45), xytext=(1.3, 0.45), arrowprops=dict(arrowstyle="<->", lw=0.7))
ax.text(2.2, 0.25, "centros a más de 1 km", ha="center", va="top", fontsize=7)
ax.text(2.2, 3.75, "terreno compartido:\nlos dos sitios se unen", ha="center", va="bottom", fontsize=7)
ax.set_xlim(0, 4.4); ax.set_ylim(-0.3, 5.0)
titulos.append("A2. Superposición: sitios\nque comparten terreno"); ejes_fig.append(ax)

# ---- B: destino de las imágenes y rotación de pliegues ----
ax = fig.add_subplot(gs[2]); ax.axis("off")
for v in range(k_pl):
    y = k_pl - 1 - v
    ax.text(-0.15, y + 0.4, f"Vuelta {v + 1}", ha="right", va="center", fontsize=7)
    for k in range(k_pl):
        ax.add_patch(Rectangle((k, y), 0.92, 0.8, fc=C_VAL if k == v else C_ENT, ec="white"))
    ax.add_patch(Rectangle((k_pl + 0.25, y), 0.92, 0.8, fc=C_HO, ec="white", alpha=0.30))
for k, n in enumerate(tam.values):
    ax.text(k + 0.46, k_pl + 0.05, f"P{k + 1}\n{n}", ha="center", va="bottom", fontsize=6.5)
ax.text(k_pl + 0.71, k_pl + 0.05, f"Holdout\n{n_ho}", ha="center", va="bottom", fontsize=6.5)
for k in range(k_pl):
    ax.add_patch(Rectangle((k, -1.3), 0.92, 0.8, fc=C_ENT, ec="white"))
ax.add_patch(Rectangle((k_pl + 0.25, -1.3), 0.92, 0.8, fc=C_HO, ec="white"))
ax.text(-0.15, -0.9, "Modelo final", ha="right", va="center", fontsize=7)
ax.set_xlim(-1.7, k_pl + 1.3); ax.set_ylim(-1.7, k_pl + 1.1)
ax.legend(handles=[Patch(fc=C_ENT, label="entrenamiento"), Patch(fc=C_VAL, label="validación"),
                   Patch(fc=C_HO, alpha=0.30, label="holdout sin usar"), Patch(fc=C_HO, label="evaluación final")],
          loc="upper center", bbox_to_anchor=(0.45, 0.02), ncol=2, fontsize=6.5, frameon=False)
titulos.append("B. Destino de las imágenes\n(número por pliegue)"); ejes_fig.append(ax)

# Títulos de los tres paneles a la misma altura
for a, t in zip(ejes_fig, titulos):
    fig.text(a.get_position().x0, 0.99, t, ha="left", va="top", fontsize=8)

guardar(fig, "figura_04_particion")
plt.show()
print("Imágenes por pliegue:", dict(tam), "| holdout:", n_ho)

**En Word.** Inserta `figura_04_particion.png` en la sección 3.2.1, después del párrafo que la cita por primera vez. Propuesta de título y nota:

- *Esquema de la partición de los datos por sitio*
- *Nota.* A1 y A2 son esquemas: los puntos representan centros de imágenes de una misma clase y los cuadrados, sus huellas. En B, cada fila es una vuelta de la validación cruzada; los números indican las imágenes de cada pliegue y del *holdout*. Elaboración propia con el *notebook* `06_figuras_tdg`.

## 6.4 Figura B1: las alteraciones de las pruebas de contexto

Muestra, para una imagen de la clase SCD y una de la clase Jagüey, las cinco alteraciones de la sección 3.2.5 y el montaje en conflicto: la estructura de una clase pegada sobre el paisaje de la otra. Usa exactamente los parámetros del notebook 4 (Anexo B5): disco central de radio 0,30 del lado, relleno con el color medio, desenfoque gaussiano de radio 0,20 del lado y borde del montaje suavizado con radio 0,02 del lado, todo sobre copias de 448 × 448 píxeles.

Las imágenes se eligen al azar de la validación cruzada. Si prefieres otras, escribe sus `id` en `ID_SCD` e `ID_JAGUEY`.

In [ ]:
from PIL import ImageDraw, ImageFilter, ImageOps, ImageStat

LADO, R_CENTRO, R_DESENFOQUE, R_BORDE = 448, 0.30, 0.20, 0.02
ID_SCD, ID_JAGUEY = None, None      # por ejemplo: 263, 571

def disco(lado, radio):
    m = Image.new("L", (lado, lado), 0)
    ImageDraw.Draw(m).ellipse([lado / 2 - radio * lado, lado / 2 - radio * lado,
                               lado / 2 + radio * lado, lado / 2 + radio * lado], fill=255)
    return m

MASCARA = disco(LADO, R_CENTRO)
MASCARA_SUAVE = MASCARA.filter(ImageFilter.GaussianBlur(R_BORDE * LADO))
color_medio = lambda im: tuple(int(v) for v in ImageStat.Stat(im).mean)
borroso = lambda im: im.filter(ImageFilter.GaussianBlur(R_DESENFOQUE * LADO))
ALTERACIONES = {
    "Original": lambda im: im,
    "Grises": lambda im: ImageOps.grayscale(im).convert("RGB"),
    "Solo el centro": lambda im: Image.composite(im, Image.new("RGB", im.size, color_medio(im)), MASCARA),
    "Solo el entorno": lambda im: Image.composite(Image.new("RGB", im.size, color_medio(im)), im, MASCARA),
    "Centro\ndesenfocado": lambda im: Image.composite(borroso(im), im, MASCARA),
    "Entorno\ndesenfocado": lambda im: Image.composite(im, borroso(im), MASCARA),
}

def cargar(ident):
    with Image.open(part.loc[part["id"] == ident, "ruta"].iloc[0]) as im:
        return im.convert("RGB").resize((LADO, LADO), Image.LANCZOS)

rng = np.random.default_rng(SEMILLA)
cv = part[part["split"] == "cv"]
elegir = lambda clase: int(cv.loc[cv["clase"] == clase, "id"].iloc[int(rng.integers((cv["clase"] == clase).sum()))])
id_scd = ID_SCD or elegir("H2")
id_jag = ID_JAGUEY or elegir("Jaguey")
im_scd, im_jag = cargar(id_scd), cargar(id_jag)
filas = [("SCD", im_scd, im_jag), ("Jagüey", im_jag, im_scd)]

n_col = len(ALTERACIONES) + 1
fig, ejes = plt.subplots(2, n_col, figsize=(ANCHO, ANCHO * 0.33))
for f, (nombre_fila, im, otra) in enumerate(filas):
    for c, (titulo, alterar) in enumerate(ALTERACIONES.items()):
        ejes[f, c].imshow(alterar(im))
        if f == 0:
            ejes[f, c].set_title(titulo, fontsize=6.5, pad=2)
    ejes[f, -1].imshow(Image.composite(im, otra, MASCARA_SUAVE))
    if f == 0:
        ejes[f, -1].set_title("Montaje en\nconflicto", fontsize=6.5, pad=2)
    ejes[f, 0].set_ylabel(nombre_fila, fontsize=7.5, fontweight="bold")
for ax in ejes.ravel():
    ax.set_xticks([]); ax.set_yticks([])
    for lado in ax.spines.values():
        lado.set_visible(True); lado.set_linewidth(0.4)
fig.tight_layout(w_pad=0.25, h_pad=0.3)
guardar(fig, "figura_B1_contexto")
plt.show()
print(f"id usados: SCD {id_scd}, Jagüey {id_jag}")

**En Word.** Va en el Anexo B, al final de B5, con el rótulo nuevo **«Figura B»** (como hiciste con «Tabla B»: borra el espacio entre la «B» y el número). Añade al párrafo de B5 la frase «La Figura B1 muestra las alteraciones y el montaje aplicados a una imagen de cada clase.». Propuesta de título y nota:

- *Alteraciones de las pruebas de contexto aplicadas a una imagen de la clase SCD y una de la clase Jagüey*
- *Nota.* En el montaje en conflicto, el disco central de cada imagen se pegó sobre el entorno de la otra. Parámetros en el Anexo B5. Imágenes de Google Maps Static API. Elaboración propia con el *notebook* `06_figuras_tdg`.

# Figuras de los resultados

Las secciones siguientes solo leen archivos de `salidas/` (no abren imágenes ni usan GPU).

## 6.5 Figura 5: *zoom* por clase y exactitud balanceada de las líneas base y de las redes

- **A.** Número de imágenes de cada clase en cada nivel de *zoom* (las 600 imágenes).
- **B.** Exactitud balanceada en la validación cruzada: el azar, las tres líneas base y las tres redes.

In [ ]:
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from matplotlib.colors import LinearSegmentedColormap

ORDEN_CLASES = ["H2", "Jaguey", "Fairy circle", "Fairy fort", "Farm circle",
                "Flooded dune", "Impact crater", "Karst", "Salt lake", "Thermokarst"]
AZULES = LinearSegmentedColormap.from_list("azules", ["#FFFFFF", OKABE["azul"]])

lineas = pd.read_csv(OUT_DIR / "lineas_base.csv", sep=";", encoding="utf-8-sig").set_index("linea_base")["exactitud_balanceada"]
oof = pd.read_csv(OUT_DIR / "oof_predicciones.csv", sep=";", encoding="utf-8-sig")
redes = oof.groupby("modelo").apply(lambda g: balanced_accuracy_score(g["clase"], g["pred"]))

fig, (axA, axB) = plt.subplots(1, 2, figsize=(ANCHO, ANCHO * 0.42), gridspec_kw={"width_ratios": [1.45, 1]})

tabla = pd.crosstab(part["clase"], part["zoom"]).reindex(ORDEN_CLASES)
axA.imshow(tabla.values, cmap=AZULES, aspect="auto")
axA.set_xticks(range(tabla.shape[1]), tabla.columns)
axA.set_yticks(range(len(tabla)), [nombre(c) for c in tabla.index])
for (i, j), v in np.ndenumerate(tabla.values):
    if v:
        axA.text(j, i, v, ha="center", va="center", fontsize=6.5, color="white" if v > 0.55 * tabla.values.max() else "black")
axA.set_xlabel("Nivel de zoom")
axA.tick_params(length=0)
for s in axA.spines.values(): s.set_visible(False)
axA.set_title("A. Imágenes por clase y nivel de zoom", fontsize=8, loc="left")

etiquetas = {"solo zoom": "Línea base: zoom", "solo color y contraste": "Línea base: color",
             "zoom + color": "Línea base: zoom + color",
             "yolov8n-cls": "YOLOv8n-cls", "yolov8s-cls": "YOLOv8s-cls", "resnet18": "ResNet-18"}
nombres_b = ["Azar"] + [etiquetas[k] for k in lineas.index] + [etiquetas[k] for k in ["yolov8n-cls", "yolov8s-cls", "resnet18"]]
valores_b = [0.10] + list(lineas.values) + [redes[k] for k in ["yolov8n-cls", "yolov8s-cls", "resnet18"]]
colores_b = [OKABE["gris"]] + [OKABE["naranja"]] * len(lineas) + [OKABE["azul"]] * 3
y = np.arange(len(nombres_b))[::-1]
axB.barh(y, valores_b, color=colores_b, height=0.65)
for yi, v in zip(y, valores_b):
    axB.text(v + 0.015, yi, f"{v:.2f}".replace(".", ","), va="center", fontsize=6.5)
axB.set_yticks(y, nombres_b)
axB.set_xlim(0, 1.08); axB.set_xlabel("Exactitud balanceada"); axB.xaxis.set_major_formatter(coma_decimal)
axB.set_title("B. Validación cruzada", fontsize=8, loc="left")
fig.tight_layout(w_pad=1.0)
guardar(fig, "figura_05_zoom_lineas_base")
plt.show()

**En Word (sección 4.2).** *Distribución del zoom por clase y exactitud balanceada de las líneas base y de las redes.* — *Nota.* A: las 600 imágenes. B: predicciones fuera del pliegue de la validación cruzada (499 imágenes); el azar corresponde a 0,10 con diez clases. Elaboración propia con el *notebook* `06_figuras_tdg`.

## 6.6 Figura 6: matriz de confusión del modelo final en el *holdout*

In [ ]:
ho = pd.read_csv(OUT_DIR / "holdout_predicciones.csv", sep=";", encoding="utf-8-sig")
cm = confusion_matrix(ho["clase"], ho["pred"], labels=ORDEN_CLASES)
cm_fila = cm / cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(ANCHO * 0.72, ANCHO * 0.62))
ax.imshow(cm_fila, cmap=AZULES, vmin=0, vmax=1)
nombres_c = [nombre(c) for c in ORDEN_CLASES]
ax.set_xticks(range(len(nombres_c)), nombres_c, rotation=45, ha="right")
ax.set_yticks(range(len(nombres_c)), nombres_c)
for (i, j), n in np.ndenumerate(cm):
    if n:
        ax.text(j, i, n, ha="center", va="center", fontsize=7, color="white" if cm_fila[i, j] > 0.5 else "black")
ax.set_xlabel("Clase predicha"); ax.set_ylabel("Clase real")
ax.tick_params(length=0)
for s in ax.spines.values(): s.set_visible(False)
fig.tight_layout()
guardar(fig, "figura_06_confusion_holdout")
plt.show()
print("Aciertos:", int(np.trace(cm)), "de", int(cm.sum()))

**En Word (sección 4.3).** *Matriz de confusión del modelo final en el holdout.* — *Nota.* Cada fila es una clase real; los números son imágenes y el color, la proporción dentro de la fila. 101 imágenes, de 10 u 11 por clase. Elaboración propia con el *notebook* `06_figuras_tdg`.

## 6.7 Figura 7: pruebas de contexto y montajes

- **A.** F1-macro de validación cruzada según la parte de la imagen que ve el modelo; cada alteración junto a su control desenfocado.
- **B.** Clase asignada a los montajes: estructura de una clase sobre el paisaje de la misma clase (control) o de la otra.

In [ ]:
res = json.load(open(OUT_DIR / "resumen_interpretaciones.json", encoding="utf-8"))
ctx = res["contexto_f1_macro"]
orden_ctx = [("original", "Original", OKABE["gris"]),
             ("sin color (grises)", "Grises", OKABE["purpura"]),
             ("solo el centro", "Solo el centro", OKABE["azul"]),
             ("entorno desenfocado", "Entorno desenfocado\n(control)", OKABE["celeste"]),
             ("solo el entorno", "Solo el entorno", OKABE["bermellon"]),
             ("centro desenfocado", "Centro desenfocado\n(control)", OKABE["naranja"])]
mont = res["montajes"]
orden_m = [("H2 sobre paisaje de otra H2 (control)", "SCD sobre\nSCD\n(control)"),
           ("H2 sobre paisaje guajiro", "SCD sobre\npaisaje\nguajiro"),
           ("Jagüey sobre paisaje de otro Jagüey (control)", "Jagüey sobre\njagüey\n(control)"),
           ("Jagüey sobre paisaje de una H2", "Jagüey sobre\npaisaje\nde SCD")]

fig, (axA, axB) = plt.subplots(1, 2, figsize=(ANCHO, ANCHO * 0.45), gridspec_kw={"width_ratios": [1, 1.1]})
y = np.arange(len(orden_ctx))[::-1]
vals = [ctx[k] for k, _, _ in orden_ctx]
axA.barh(y, vals, color=[c for _, _, c in orden_ctx], height=0.65)
for yi, v in zip(y, vals):
    axA.text(v + 0.01, yi, f"{v:.2f}".replace(".", ","), va="center", fontsize=6.5)
axA.axvline(ctx["original"], color="black", lw=0.6, ls=":")
axA.set_yticks(y, [t for _, t, _ in orden_ctx]); axA.set_xlim(0, 1.05)
axA.set_xlabel("F1-macro"); axA.xaxis.set_major_formatter(coma_decimal); axA.set_title("A. Alteraciones de la imagen", fontsize=8, loc="left")

x = np.arange(len(orden_m))
scd = np.array([mont[k]["predichos H2"] for k, _ in orden_m]) * 100
jag = np.array([mont[k]["predichos Jagüey"] for k, _ in orden_m]) * 100
otras = 100 - scd - jag
axB.bar(x, scd, color=OKABE["bermellon"], width=0.6, label="SCD")
axB.bar(x, jag, bottom=scd, color=OKABE["verde"], width=0.6, label="Jagüey")
axB.bar(x, otras, bottom=scd + jag, color="#D9D9D9", width=0.6, label="Otras clases")
for xi, (a, b) in enumerate(zip(scd, jag)):
    if a >= 6: axB.text(xi, a / 2, f"{a:.0f} %", ha="center", va="center", fontsize=6.5, color="white")
    if b >= 6: axB.text(xi, a + b / 2, f"{b:.0f} %", ha="center", va="center", fontsize=6.5, color="white")
axB.set_xticks(x, [t for _, t in orden_m], fontsize=6.5); axB.set_ylim(0, 100)
axB.set_ylabel("Montajes (%)"); axB.set_title("B. Clase asignada a los montajes", fontsize=8, loc="left")
axB.legend(fontsize=6.5, frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.30), ncol=3)
fig.tight_layout(w_pad=1.2)
guardar(fig, "figura_07_contexto")
plt.show()

**En Word (sección 4.4).** *Desempeño del modelo según la parte de la imagen que ve y clase asignada a los montajes.* — *Nota.* A: modelos de cada pliegue sobre sus imágenes de validación; la línea punteada es el F1-macro original. B: entre 474 y 532 montajes por tipo, dentro de cada pliegue. Elaboración propia con el *notebook* `06_figuras_tdg`.